In [ ]:
!pip install -qU torchao transformers datasets trl peft bitsandbytes accelerate wandb word2number num2words

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.4 MB/s eta 0:00:00


In [ ]:
import torch
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer
from trl import SFTConfig, SFTTrainer
from tqdm import tqdm
from word2number import w2n
import math
import numpy as np
from num2words import num2words

import wandb
wandb.login()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hathreyap (hathreyap-university-of-stuttgart) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cuda


In [ ]:
def _extract_gold_int(gold_str: str) -> int:
    """Extracts integer from GSM8K answer field (e.g., '...#### 72' or '#### 1,200')."""
    m = re.search(r"####\s*(-?\d[\d,]*)", gold_str)
    if not m:
        raise ValueError(f"Could not extract gold integer from: {gold_str!r}")
    return int(m.group(1).replace(",", ""))

_FINAL_LINE_RE = re.compile(
    r"###[ \t]+(.+?)[ \t]*\Z",
    re.MULTILINE,
)

_MAGNITUDE_WORDS = {"hundred", "thousand", "million"}

_VALID_MULTIPLIERS = {
    "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",
    "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen",
    "seventeen", "eighteen", "nineteen", "twenty", "thirty", "forty", "fifty",
    "sixty", "seventy", "eighty", "ninety",
}

def _normalise(raw: str) -> str:
    """Lowercase, strip commas, hyphens to spaces, collapse whitespace."""
    s = raw.replace(",", "").replace("-", " ").lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _needs_prepend(cleaned: str) -> bool:
    """
    Returns True if the input starts with a magnitude word that has no
    valid multiplier before it. Catches both:
      - 'thousand five' (would IndexError)
      - 'hundred and one' (would silently return 100)
    """
    if not cleaned:
        return False
    tokens = cleaned.split()
    # Walk left-to-right: find the first magnitude word and check what's before it.
    for i, tok in enumerate(tokens):
        if tok in _MAGNITUDE_WORDS:
            preceding = tokens[:i]
            # 'and' is filtered by w2n but we may have already stripped it.
            # The check is whether any preceding token is a real multiplier.
            return not any(t in _VALID_MULTIPLIERS for t in preceding)
    return False

def _extract_generated_number(completion_str: str):
    """
    Extract the answer after the final '### ' marker and parse it as
    an integer. Returns None for unparseable, malformed, or non-integer
    inputs.

    Recovery logic: if the input starts with a magnitude word ('hundred',
    'thousand', 'million', 'billion') without a multiplier in front, we
    prepend 'one' and retry. This handles two w2n quirks at once:
      - 'thousand five' raises IndexError -> 'one thousand five' -> 1005
      - 'hundred and one' silently returns 100 -> 'one hundred and one' -> 101
    """
    m = _FINAL_LINE_RE.search(completion_str.rstrip())
    if not m:
        return None

    raw = m.group(1).strip()
    if not raw:
        return None

    cleaned = _normalise(raw)
    if not cleaned:
        return None

    # Proactive repair: if the input starts with a bare magnitude, prepend 'one'.
    # This addresses both the IndexError family and the silent-wrong family.
    if _needs_prepend(cleaned):
        cleaned = "one " + cleaned

    try:
        result = w2n.word_to_num(cleaned)
    except ValueError:
        return None
    except IndexError:
        # Belt-and-braces: if our pre-flight missed something, don't crash.
        return None
    except Exception:
        # Any other w2n internal failure.
        return None

    # GSM8K answers are integers. Reject decimals from 'point' constructs.
    if not isinstance(result, int):
        return None

    return result

def plain_reward(completions, targer, **kwards):
  scores = 0.0
  for group, tgt in zip(completions, target):
    gold = _extract_gold_int(tgt)
    for rollout in group:
      text = rollout[-1]["content"] if isinstance(rollout, list) else rollout["content"]
      pred = _extract_generated_number(text)
      if pred is not None:
        yield pred == gold
      else:
        yield False


def strict_gated_reward(completions, target, **kwargs):
    scores = []
    fmt_vals, lip_vals, cor_vals = [], [], []

    for completion, tgt in zip(completions, target):
        try:
            gold = _extract_gold_int(tgt)
        except ValueError:
            # Malformed gold -> give this specific rollout 0
            scores.append(0.0)
            fmt_vals.append(0.0)
            lip_vals.append(0.0)
            cor_vals.append(0.0)
            continue

        # --- EXTRACT THE COMPLETION TEXT ---
        # If using chat templates, completion is a list of dicts. Grab the last message.
        if isinstance(completion, list) and len(completion) > 0 and isinstance(completion[-1], dict):
            text = completion[-1]["content"]
        # If using standard string formats, completion is already just the generated string.
        elif isinstance(completion, str):
            text = completion
        else:
            text = str(completion)

        # --- 1. Constraint Checks ---
        r_fmt = -2.0  # Default penalty
        parsed = None
        if _FINAL_LINE_RE.search(text.rstrip()):
            parsed = _extract_generated_number(text)
            if parsed is not None:
                r_fmt = 1.0  # Reward for good format

        digit_count = sum(ch.isdigit() for ch in text)
        if digit_count == 0:
            r_lip = 1.0
        else:
            r_lip = -2.0

        # --- 2. Gated Math Reward ---
        r_cor = 0.0
        if r_fmt > 0.0 and r_lip > 0.0:
            r_cor = 5.0 if (parsed is not None and parsed == gold) else -1.0

        # --- 3. Total Score ---
        score = r_fmt + r_lip + r_cor

        scores.append(score)
        fmt_vals.append(r_fmt)
        lip_vals.append(r_lip)
        cor_vals.append(r_cor)

    if wandb.run is not None:
      wandb.log({
          "train/rewards/r_fmt/mean": np.mean(fmt_vals),
          "train/rewards/r_fmt/std": np.std(fmt_vals),
          "train/rewards/r_lip/mean": np.mean(lip_vals),
          "train/rewards/r_lip/std": np.std(lip_vals),
          "train/rewards/r_cor/mean": np.mean(cor_vals),
          "train/rewards/r_cor/std": np.std(cor_vals),
          # "train/rewards/r_all/mean": np.mean(all_vals),
          # "train/rewards/r_all/std": np.std(all_vals),
      }, commit=False)

    return scores

In [ ]:
# ---------- Data loading and transformation ----------

dataset_id = "openai/gsm8k"
dataset_config = "main"

dataset = load_dataset(dataset_id, dataset_config)
train_dataset = dataset["train"]
test_dataset = dataset["test"].select(range(200))


# ---------- Transformation utilities ----------

DIGIT_RE = re.compile(r'\d+(?:,\d{3})*')
ANNOTATION_RE = re.compile(r'<<[^>]*>>')

def clean_reasoning(text):
    """
    Remove GSM8K calculation annotations (<<a op b = c>>) and
    replace all remaining digits with English words.
    """
    # Step 1: strip the <<...>> markers entirely
    text = ANNOTATION_RE.sub('', text)
    # Step 2: collapse any double spaces left behind
    text = re.sub(r'  +', ' ', text)
    # Step 3: replace remaining digits with words
    text = replace_digits_with_words(text)
    return text


def replace_digits_with_words(text):
    """Replace every digit-string in `text` with its English-word form."""
    def convert(match):
        num = int(match.group(0).replace(',', ''))
        return num2words(num)
    return DIGIT_RE.sub(convert, text)


def transform_gsm8k_answer(answer_text):
    """
    Transform a GSM8K answer into a digit-free, ###-formatted version.
    Returns None for malformed examples.
    """
    parts = answer_text.split('####')
    if len(parts) != 2:
        return None

    reasoning = parts[0].rstrip()
    final_int_str = parts[1].strip().replace(',', '')

    try:
        final_int = int(final_int_str)
    except ValueError:
        return None

    reasoning = clean_reasoning(reasoning)
    reasoning_word_form = replace_digits_with_words(reasoning)
    final_words = num2words(final_int)
    return f"{reasoning_word_form}\n### {final_words}"


# ---------- Prompt format ----------
# Minimal format used by both SFT (training) and eval. The model learns
# this structure during SFT; no few-shots needed at inference time.

def make_prompt(question):
    return f"Question:\n{question}\nAnswer:\n"


# ---------- Training-set formatting ----------

def format_for_sft(example):
    """
    For SFT: produce {prompt, completion} where prompt is the question
    in our minimal format and completion is the digit-free transformed answer.
    Malformed examples get None and are filtered out below.
    """
    completion = transform_gsm8k_answer(example["answer"])
    if completion is None:
        return {"prompt": None, "completion": None}

    return {
        "prompt": make_prompt(example["question"]),
        "completion": completion,
        "target": example["answer"]
    }


train_dataset = train_dataset.map(
    format_for_sft,
    remove_columns=train_dataset.column_names,
)
# Drop malformed examples (those with completion=None)
train_dataset = train_dataset.filter(lambda x: x["completion"] is not None)


# ---------- Test-set formatting ----------
# IMPORTANT: do NOT transform the test answers. Eval needs the original
# gold answer to score against. We only reformat the question side to
# match the SFT prompt format, and pass the original answer through as `target`.

def format_for_eval(example):
    return {
        "prompt": make_prompt(example["question"]),
        "target": example["answer"],   # original gold, unchanged
    }


test_dataset = test_dataset.map(
    format_for_eval,
    remove_columns=test_dataset.column_names,
)


# ---------- Sanity checks ----------
# Verify the data is what we expect before training.

print(f"Train examples: {len(train_dataset)}")
print(f"Train columns: {train_dataset.column_names}")
print(f"Test examples: {len(test_dataset)}")
print(f"Test columns: {test_dataset.column_names}")
print()

# Inspect one training example
example = train_dataset[0]
print("=== Training example 0 ===")
print(f"Prompt: {example['prompt']!r}")
print(f"Completion: {example['completion']!r}")

# Verify the transformation actually produces digit-free output
digit_count = sum(c.isdigit() for c in example["completion"])
print(f"Digits in completion: {digit_count} (must be 0)")
assert digit_count == 0, "Transformation left digits in the completion!"

# Verify the completion parses through the verifier
parsed = _extract_generated_number(example["completion"])
print(f"Parsed answer from completion: {parsed}")

# Verify the parsed answer matches the original gold (before transformation)
# Note: train_dataset has lost the original answer column at this point;
# we'd have to re-load to check this. Instead, spot-check a few more examples
# for the parseable+digit-free invariant.
print()
print("=== Spot-check: 5 random training examples ===")
import random
for idx in random.sample(range(len(train_dataset)), 5):
    ex = train_dataset[idx]
    digits = sum(c.isdigit() for c in ex["completion"])
    parsed = _extract_generated_number(ex["completion"])
    status = "OK" if digits == 0 and parsed is not None else "FAIL"
    print(f"  [{status}] idx={idx} digits={digits} parsed={parsed}")

print()
print("=== Test example 0 ===")
test_ex = test_dataset[0]
print(f"Prompt: {test_ex['prompt']!r}")
print(f"Target (first 80 chars): {test_ex['target'][:80]!r}")
gold = _extract_gold_int(test_ex["target"])
print(f"Gold answer: {gold}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Train examples: 7473
Train columns: ['prompt', 'completion', 'target']
Test examples: 200
Test columns: ['prompt', 'target']

=== Training example 0 ===
Prompt: 'Question:\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAnswer:\n'
Completion: 'Natalia sold forty-eight/two = twenty-four clips in May.\nNatalia sold forty-eight+twenty-four = seventy-two clips altogether in April and May.\n### seventy-two'
Digits in completion: 0 (must be 0)
Parsed answer from completion: 72

=== Spot-check: 5 random training examples ===
  [OK] idx=5837 digits=0 parsed=32
  [OK] idx=6564 digits=0 parsed=120
  [OK] idx=5495 digits=0 parsed=14
  [OK] idx=5556 digits=0 parsed=450
  [OK] idx=3611 digits=0 parsed=66

=== Test example 0 ===
Prompt: "Question:\nJanet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells th

In [ ]:
sample_answer = dataset["train"][0]["answer"]
print("Original:")
print(sample_answer)
print()
print("Transformed:")
print(transform_gsm8k_answer(sample_answer))

Original:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

Transformed:
Natalia sold forty-eight/two = twenty-four clips in May.
Natalia sold forty-eight+twenty-four = seventy-two clips altogether in April and May.
### seventy-two


In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "right"
# Ensure padding token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map=device
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
    lora_dropout=0.05
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 3,858,432 || all params: 497,891,200 || trainable%: 0.7750


In [ ]:
print("Columns:", train_dataset.column_names)
example = train_dataset[0]
print("Prompt:", repr(example["prompt"]))
print("Completion:", repr(example["completion"]))
print("Digits in completion:", sum(c.isdigit() for c in example["completion"]))
print("Parses to:", _extract_generated_number(example["completion"]))

Columns: ['prompt', 'completion', 'target']
Prompt: 'Question:\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAnswer:\n'
Completion: 'Natalia sold forty-eight/two = twenty-four clips in May.\nNatalia sold forty-eight+twenty-four = seventy-two clips altogether in April and May.\n### seventy-two'
Digits in completion: 0
Parses to: 72


In [ ]:
# ---------- Pre-training verification ----------
# Confirm the chat template / tokenization works correctly on real data
# before spending compute on training.

example = train_dataset[0]
prompt_ids = tokenizer(example["prompt"])["input_ids"]
full_ids = tokenizer(example["prompt"] + example["completion"])["input_ids"]

print(f"Prompt tokens: {len(prompt_ids)}")
print(f"Full tokens (prompt + completion): {len(full_ids)}")
print(f"Completion tokens (loss should be on these): {len(full_ids) - len(prompt_ids)}")

# The first len(prompt_ids) tokens of full_ids should match prompt_ids exactly.
# If they don't, the tokenizer is merging tokens across the boundary, which
# would make completion-only loss masking off by one.
boundary_clean = full_ids[:len(prompt_ids)] == prompt_ids
print(f"Tokenization boundary clean: {boundary_clean}")
if not boundary_clean:
    print("WARNING: token merging at boundary — completion-only masking may be slightly off.")

# Verify the trainable params are what we expect
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


# ---------- SFT configuration ----------

sft_config = SFTConfig(
    output_dir="qwen-gsm8k-sft",

    # Optimization
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # Training schedule
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # Sequence handling
    max_length=512,            # plenty for question + transformed answer
    packing=False,              # keep examples as-is; clean train-eval distribution
    completion_only_loss=True,  # only compute loss on completion tokens, not the prompt

    # Logging
    logging_steps=10,
    report_to="wandb",

    # Checkpointing — frequent enough to recover from disconnects
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    save_only_model=False,

    # Precision
    bf16=True,
    remove_unused_columns=False,

    # Reproducibility
    seed=42,
)


# ---------- Build the trainer ----------

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

# Verify completion-only loss is actually on
# (older trl versions ignore the config flag silently)
print(f"\ncompletion_only_loss: {sft_config.completion_only_loss}")


# ---------- Train ----------

trainer.train()


# ---------- Save the LoRA adapter ----------

save_path = "./sft_lora_weights"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)   # save tokenizer alongside for reproducibility
print(f"\nSFT-LoRA weights saved to {save_path}")


# ---------- Switch padding side back for eval ----------

tokenizer.padding_side = "left"
print(f"Tokenizer padding side reset to: {tokenizer.padding_side}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Prompt tokens: 43
Full tokens (prompt + completion): 80
Completion tokens (loss should be on these): 37
Tokenization boundary clean: True

Trainable params: 3,858,432 / 497,891,200 (0.77%)


Adding EOS to train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



completion_only_loss: True


Step,Training Loss
10,2.030487
20,2.169094
30,2.080913
40,1.980343
50,1.974014
60,1.827875
70,1.737574
80,1.585345
90,1.427836
100,1.292203



SFT-LoRA weights saved to ./sft_lora_weights
Tokenizer padding side reset to: left


In [ ]:
from peft import PeftModel
import os
import zipfile
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Unzip the weights
zip_path = "/content/sft_lora-weights.zip"
extract_path = "./content/sft_lora_weights"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

# Check what got extracted — sometimes there's a nested directory
print(os.listdir(extract_path))
# You're looking for a directory containing adapter_config.json and adapter_model.safetensors
# (or adapter_model.bin on older peft versions)

# If everything is directly in extract_path, use it. If there's a nested folder,
# use that nested folder as the adapter path.
adapter_path = extract_path  # adjust if you see a subdirectory in the listing

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map=device,
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()

# Now add a fresh LoRA for GRPO training on top of the merged model
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
tokenizer.padding_side = "left"

['tokenizer.json', 'chat_template.jinja', 'training_args.bin', 'README.md', 'content', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer_config.json']


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
training_args = GRPOConfig(
    output_dir="qwen-gsm8k-lipogram",
    learning_rate=3e-6,
    lr_scheduler_type="cosine",
    logging_steps=1,
    max_steps=300, # Adjust based on 4-hour limit
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # --- YOUR HYPERPARAMETERS TO DEFEND ---
    temperature=0.7,
    top_p=0.95,
    max_completion_length=512,
    num_generations=16,

    beta=0.01,

    # --- REQUIRED CONSTRAINTS ---
    use_vllm=False, # Strictly required for Colab T4

    report_to="wandb",

    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    save_only_model=False,

)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[strict_gated_reward],
    args=training_args,
    train_dataset=train_dataset,
)

# Start training
trainer.train()

save_path = "./lora_weights"
trainer.save_model(save_path)
print(f"LoRA weights saved to {save_path}")

wandb.finish()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.574633
2,-0.067404
3,0.057918
4,0.000007
5,-0.028988
6,0.079040
7,-0.011812
8,0.000010
9,0.000010
10,-0.000729


Step,Training Loss
1,0.574633
2,-0.067404
3,0.057918
4,0.000007
5,-0.028988
6,0.079040
7,-0.011812
8,0.000010
9,0.000010
10,-0.000729


In [ ]:
# @title
import os
import zipfile
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Unzip the weights
zip_path = "/content/sft_lora-weights.zip"
extract_path = "./sft_lora-weights/content/sft_lora_weights"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

# Check what got extracted — sometimes there's a nested directory
print(os.listdir(extract_path))
# You're looking for a directory containing adapter_config.json and adapter_model.safetensors
# (or adapter_model.bin on older peft versions)

# If everything is directly in extract_path, use it. If there's a nested folder,
# use that nested folder as the adapter path.
adapter_path = extract_path  # adjust if you see a subdirectory in the listing


# 2. Load the base model fresh
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
)


# 3. Apply the LoRA adapter on top of the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print("Loaded successfully.")

['adapter_config.json', 'content', 'tokenizer.json', 'chat_template.jinja', 'training_args.bin', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors']


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded successfully.


In [ ]:
!zip -r /content/sft_weights.zip /content/sft_lora_weights/

  adding: content/sft_lora_weights/ (stored 0%)
  adding: content/sft_lora_weights/tokenizer.json (deflated 81%)
  adding: content/sft_lora_weights/chat_template.jinja (deflated 71%)
  adding: content/sft_lora_weights/training_args.bin (deflated 53%)
  adding: content/sft_lora_weights/README.md (deflated 65%)
  adding: content/sft_lora_weights/adapter_config.json (deflated 59%)
  adding: content/sft_lora_weights/adapter_model.safetensors (deflated 7%)
  adding: content/sft_lora_weights/tokenizer_config.json (deflated 59%)


In [ ]:
from google.colab import files
files.download("/content/sft_weights.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ---------- Eval setup ----------
# Make sure tokenizer is in eval-mode padding orientation. SFT used right-padding;
# batched generation needs left-padding so all rows start generating from real
# prompt content rather than padding tokens.

tokenizer.padding_side = "left"


def evaluate_model(eval_model, dataset, tokenizer, batch_size=32):
    """
    Evaluates a model against the dataset in batches and returns a dict of counts.

    Expects each example to have:
      - 'prompt': raw text prompt (matches the SFT training format)
      - 'target': original GSM8K answer string ending in '#### <int>'
    """
    eval_model.eval()

    total = len(dataset)
    correct_math = 0
    correct_lipogram = 0
    correct_format = 0
    correct_all = 0

    generation_kwargs = {
        "max_new_tokens": 512,
        "do_sample": False,
        "temperature": 0.0,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    print(f"Evaluating {total} examples in batches of {batch_size}...")

    for i in tqdm(range(0, total, batch_size)):
        batch = dataset.select(range(i, min(i + batch_size, total)))

        # Use raw prompts directly — no chat template
        prompt_texts = [ex["prompt"] for ex in batch]
        inputs = tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(eval_model.device)

        with torch.no_grad():
            outputs = eval_model.generate(**inputs, **generation_kwargs)

        # Decode only the newly-generated tokens (everything after the padded prompt).
        # With left-padding, all rows in the batch share the same input length
        # (the padded length), so slicing at inputs.input_ids.shape[1] is correct.
        input_length = inputs.input_ids.shape[1]

        for j in range(len(batch)):
            completion = tokenizer.decode(
                outputs[j][input_length:], skip_special_tokens=True
            )
            target_str = batch[j]["target"]

            # 1. Format: ends with "### <english-words>"
            format_pass = bool(
                re.search(r"###\s+[a-zA-Z][a-zA-Z\- ]*\s*$", completion)
            )

            # 2. Lipogram: zero ASCII digits anywhere
            lipogram_pass = not any(c.isdigit() for c in completion)

            # 3. Correctness: parsed answer matches gold
            try:
                gold_num = _extract_gold_int(target_str)
            except ValueError:
                gold_num = None
            gen_num = _extract_generated_number(completion)
            math_pass = (
                gold_num is not None
                and gen_num is not None
                and gold_num == gen_num
            )

            all_pass = format_pass and lipogram_pass and math_pass

            if math_pass:
                correct_math += 1
            if lipogram_pass:
                correct_lipogram += 1
            if format_pass:
                correct_format += 1
            if all_pass:
                correct_all += 1

    return {
        "numeric_correctness": correct_math,
        "no_digits_compliance": correct_lipogram,
        "format_compliance": correct_format,
        "all_three": correct_all,
    }


# ---------- Sanity check on a single example before full eval ----------
# Confirms generation works end-to-end and produces something reasonable.

print("=== Sanity check ===")
sample = test_dataset[0]
inputs = tokenizer(sample["prompt"], return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.pad_token_id,
    )
completion = tokenizer.decode(
    out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
)
print(f"Prompt: {sample['prompt']!r}")
print(f"Completion:\n{completion}")
print(f"Digits in completion: {sum(c.isdigit() for c in completion)}")
print(f"Parsed answer: {_extract_generated_number(completion)}")
print(f"Gold: {_extract_gold_int(sample['target'])}")
print()


# ---------- Run eval on base and trained models ----------

print("Evaluating Base Model...")
with model.disable_adapter():
    base_stats = evaluate_model(model, test_dataset, tokenizer, batch_size=32)

print("\nEvaluating Trained Model...")
trained_stats = evaluate_model(model, test_dataset, tokenizer, batch_size=32)


# ---------- Build and save the eval table ----------

total = len(test_dataset)
table_lines = [
    "### eval_table.md",
    "",
    "| Metric | Base | Trained |",
    "|---|---|---|",
    f"| numeric correctness (matches gold) | {base_stats['numeric_correctness']} | {trained_stats['numeric_correctness']} |",
    f"| no-digits compliance | {base_stats['no_digits_compliance']} | {trained_stats['no_digits_compliance']} |",
    f"| format compliance | {base_stats['format_compliance']} | {trained_stats['format_compliance']} |",
    f"| all three together | {base_stats['all_three']} | {trained_stats['all_three']} |",
    "",
    f"_Counts out of {total} examples._",
    "",
    "**Decoding configuration:**",
    "- `temperature=0.0`, `do_sample=False` (greedy)",
    "- `max_new_tokens=512`",
    "- `padding_side='left'` for batched generation",
    "- Raw prompt format (no chat template): `Question:\\n<question>\\nAnswer:\\n`",
    "- Same prompt and decoding settings used for both base and trained models.",
]
table_text = "\n".join(table_lines)

print()
print(table_text)

output_path = "eval_table.md"
with open(output_path, "w") as f:
    f.write(table_text)
print(f"\nWrote table to {output_path}")

try:
    from google.colab import files
    files.download(output_path)
    print(f"Downloading {output_path} to your machine...")
except ImportError:
    print(f"(Not running in Colab — file saved locally at {output_path}.)")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


=== Sanity check ===
Prompt: "Question:\nJanet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nAnswer:\n"
Completion:
She.U.P.GSLARCOMANGERS/TAD/20000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
Digits in completion: 500
Parsed answer: None
Gold: 18

Evaluating Base Model...
Evaluating 200 examples in batches of 32...


100%|██████████| 7/7 [05:39<00:00, 48.56s/it]



Evaluating Trained Model...
Evaluating 200 examples in batches of 32...


100%|██████████| 7/7 [02:20<00:00, 20.02s/it]


### eval_table.md

| Metric | Base | Trained |
|---|---|---|
| numeric correctness (matches gold) | 0 | 30 |
| no-digits compliance | 0 | 200 |
| format compliance | 0 | 171 |
| all three together | 0 | 30 |

_Counts out of 200 examples._

**Decoding configuration:**
- `temperature=0.0`, `do_sample=False` (greedy)
- `max_new_tokens=512`
- `padding_side='left'` for batched generation
- Raw prompt format (no chat template): `Question:\n<question>\nAnswer:\n`
- Same prompt and decoding settings used for both base and trained models.

Wrote table to eval_table.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("=== Base model rollouts ===")
with model.disable_adapter():
    for i in range(5):
        sample = test_dataset[i]
        inputs = tokenizer(sample["prompt"], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id,
            )
        completion = tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        )
        print(f"--- Example {i} ---")
        print(f"Prompt: {sample['prompt']!r}")
        print(f"Completion: {completion!r}")
        print(f"Length: {len(completion)} chars, digits={sum(c.isdigit() for c in completion)}")
        print()

=== Base model rollouts ===
--- Example 0 ---
Prompt: "Question:\nJanet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nAnswer:\n"
Completion: "To find out how many eggs Janet makes at the farmers' market each day, we need to subtract the number of eggs she eats from the total number of eggs she lays.\n\nTotal number of eggs laid by the ducks = 16\nNumber of eggs eaten in a day = 3 (breakfast) + 4 (baking muffins) = 7\n\nNumber of eggs left after breakfast and baking muffins = 16 - 7 = 9\n\nPrice per egg = $2\n\nTotal earnings per day = Number of eggs left * Price per egg = 9 * $2 = $18\n\nTherefore, Janet makes $18 at the farmers' market each day."
Length: 501 chars, digits=16

--- Example 1 ---
Prompt: 'Question:\nA robe takes 2 bolts of blue fiber and half th

In [ ]:
for i in range(5):
        sample = test_dataset[i]
        inputs = tokenizer(sample["prompt"], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id,
            )
        completion = tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        )
        print(f"--- Example {i} ---")
        print(f"Prompt: {sample['prompt']!r}")
        print(f"Completion: {completion!r}")
        print(f"Length: {len(completion)} chars, digits={sum(c.isdigit() for c in completion)}")
        print()

--- Example 0 ---
Prompt: "Question:\nJanet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nAnswer:\n"
Completion: 'She has sixteen x two = thirty-two eggs to sell.\nSo, she makes a profit of twenty - three - four = twelve dollars per day.\n### twelve'
Length: 133 chars, digits=0

--- Example 1 ---
Prompt: 'Question:\nA robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?\nAnswer:\n'
Completion: 'It takes two/two=one bolt of blue fiber\nAnd it takes one/two=sixty/ten=four bolts of white fiber\nSo a total of one+four=eight bolts\n### eight'
Length: 141 chars, digits=0

--- Example 2 ---
Prompt: 'Question:\nJosh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  Thi